# Stage 2 Notebook 74 - CULane lane-only pretraining

**Priority 3.** Pretrain the joint model's BACKBONE on CULane (88K images, lane-only, no det conflict, 8 epochs) so it has good lane-aware features BEFORE we joint-train on BDD100K.

Pipeline:
1. **Extract CULane archives** from `MyDrive/EcoCAR/downloads/CULane/` to `/content/CULane/`.
   - list.tar.gz (split files)
   - annotations_new.tar.gz (.lines.txt labels)
   - driver_<X>_<Y>frame.tar.gz (images; we extract ALL six)
2. **Train lane-only** (lambda_det=0, --allow-empty-det-labels) for 8 epochs at batch=32.
3. **Export backbone_pretrained.pt** -- the training script now saves a backbone-only checkpoint
   automatically. We'll use this in NB75 as the joint-training init.

Note: CULane has different image dimensions (590x1640 vs BDD 720x1280). Both are resized to 384x640 by the dataloader so this is a noop at the network level. CULane has max_lanes=4 (vs BDD max_lanes=10) which means the lane HEAD outputs 192 priors / 4 GTs vs 192 priors / 10 GTs during pretrain. The BACKBONE features are what we keep -- we don't transfer the lane head.

### Run mode
1. Extract CULane archives (~40 GB; may take 30-60 min).
2. Smoke.
3. 8 epochs CULane train (~2-3 hr at batch=32 with speed flags).
4. Backbone checkpoint is auto-saved as `backbone_pretrained.pt` in the work dir.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [ ]:
# Step 1: extract CULane archives.
#
# v3 (after the first run crashed with FileNotFoundError on missing .jpgs):
# the prep script now AUTO-REPAIRS partial extractions. For each driver
# archive, it counts .jpg files actually on disk and compares against
# CULane's published per-driver counts. If a marker dir has < 90 percent
# of expected jpgs, it re-extracts on top (filling gaps). The post-run
# integrity report prints the ratio so you can see which drivers were
# repaired.
#
# If you want to FORCE a clean re-extract of everything (rare; useful
# when prior extractions corrupted file metadata), set FORCE=True below.
from pathlib import Path
import os, sys, subprocess

CULANE_SRC = '/content/drive/MyDrive/EcoCAR/downloads/CULane'
CULANE_DST = '/content/CULane'
FORCE = False  # set True only if 90%-threshold logic doesn't fix the gaps

print(f'[diagnostic] listing {CULANE_SRC} ...')
try:
    for f in sorted(Path(CULANE_SRC).iterdir()):
        try:
            print(f'  {f.name}  ({f.stat().st_size/1e9:.2f} GB)')
        except OSError as e:
            print(f'  {f.name}  (stat err: {e})')
except Exception as e:
    print(f'[diagnostic] cannot list {CULANE_SRC}: {e}')
    parent = '/content/drive/MyDrive/EcoCAR/downloads'
    try:
        print(f'[diagnostic] parent contents of {parent}:')
        for f in sorted(Path(parent).iterdir()):
            print(f'  {f.name}')
    except Exception as e2:
        print(f'[diagnostic] cannot list parent: {e2}')

cmd = [sys.executable, '-u', 'stage2/scripts/prepare_culane_dataset.py',
       '--src', CULANE_SRC,
       '--dest', CULANE_DST]
if FORCE:
    cmd.append('--force')
LOG_FILE = os.path.join(LOG_DIR, 'culane_prepare.log')
print('Extracting CULane archives...  (force=' + str(FORCE) + ')')
run_streaming(cmd, log_path=LOG_FILE)

In [ ]:
# Step 2: train lane-only on CULane for 8 epochs.
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp69_rmt_gca_culane_lane_only_pretrain.yaml'

DEBUG_MODE = False
if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 8
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'culane8'
    EPOCHS = 8
    BATCH_SIZE = 32
    LIMIT_TRAIN = None
    LIMIT_VAL = 1000
    PRINT_EVERY = 100

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    # No curve-tar; the CULane data is already extracted to CULANE_DST.
    '--curve-root', '/content/CULane',
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--print-every', str(PRINT_EVERY),
    # CULane has no detection labels.
    '--allow-empty-det-labels',
    # Speed flags.
    '--workers', '6',
    '--prefetch-factor', '4',
    '--torch-compile',
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('Training CULane lane-only pretrain. Backbone checkpoint will be saved at:')
print(f'  {WORK_DIR}/backbone_pretrained.pt')
print('and copied into the output tar.')
print('About to run:', ' '.join(cmd), flush=True)
run_streaming(cmd, log_path=LOG_FILE)

## What to watch in NB74 (v3 auto-repair flow)

After cell 3 finishes you should see an `[integrity] per-driver jpg counts` block near the end of its output. Every driver should show `[OK]`. If any show `[PARTIAL]`, re-run cell 3 (the script will re-extract those drivers only). If `[MISSING]`, verify the corresponding tar.gz is in your Drive folder.

Cell 4 trains the lane head on CULane. With v3 of the training script (dataset filter), it tolerates a small number of missing .jpg files: the dataset prints `dropped N / M samples because their .jpg files do not exist on disk` at init time and continues with the surviving N. If N drops below say 80k (out of 88880 total), the prep step had bigger issues -- re-run cell 3 with FORCE=True.

Pass criteria at epoch 8 (CULane lane-only):
- `val/matched_line_iou >= 0.55` (CULane is cleaner than BDD; should converge fast).
- `val/lane_best_f1 >= 0.50` -- CULane has simple highway lanes; this is achievable.
- `val/lane/decoded_f1 >= 0.30`.
- A `best.pt` and `backbone_pretrained.pt` get saved into the output tar so NB77 can use them as the teacher / backbone init.